In [1]:
# ============================================================
# VRU Spectral Pairing Ablation — single-cell version
#
# Question: Is 4/π special, or is the reciprocal pairing (k, 1/k) the mechanism?
#
# Sweep A — paired (k, 1/k) across k ∈ [0.9, 1.75]. Peak or plateau?
# Sweep B — fixed k_h = 4/π, vary k_c independently. Does breaking reciprocal collapse?
# ============================================================

import torch
import torch.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import dataclass, field
import time

# ---------- Setup ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(42); np.random.seed(42)

PHI = 4.0 / math.pi    # 1.27324
ALPHA = math.pi / 4.0  # 0.78540
print(f'PHI = 4/π  = {PHI:.6f}')
print(f'ALPHA = π/4 = {ALPHA:.6f}')
print(f'PHI * ALPHA = {PHI*ALPHA:.6f}')

# ---------- Lorenz attractor (chaotic long-range gradient stress) ----------
def generate_lorenz(batch, seq_len, dt=0.01, device='cuda'):
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    state = torch.randn(batch, 3, device=device) * 0.5 + torch.tensor([1.0, 1.0, 1.0], device=device)
    trajectory = torch.zeros(batch, seq_len + 1, 3, device=device)
    trajectory[:, 0] = state
    def f(s):
        x, y, z = s[..., 0], s[..., 1], s[..., 2]
        return torch.stack([sigma*(y-x), x*(rho-z)-y, x*y-beta*z], dim=-1)
    for t in range(seq_len):
        k1 = f(state)
        k2 = f(state + 0.5*dt*k1)
        k3 = f(state + 0.5*dt*k2)
        k4 = f(state + dt*k3)
        state = state + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
        trajectory[:, t+1] = state
    trajectory = trajectory / 30.0
    return trajectory[:, :-1], trajectory[:, 1:]

# ---------- Paired-spectral RNN ----------
# h_{t+1} = tanh(W_x x + k_h * W_h h + k_c * W_c h)
# Two recurrent paths, independent scalar multipliers. Orthogonal init so
# starting spectra match across runs — only the scalars differ.

class PairedSpectralCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, k_h, k_c):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.k_h, self.k_c = k_h, k_c
        self.W_x = nn.Linear(input_dim, hidden_dim, bias=True)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_c = nn.Linear(hidden_dim, hidden_dim, bias=False)
        nn.init.orthogonal_(self.W_h.weight)
        nn.init.orthogonal_(self.W_c.weight)
    def forward(self, x, h):
        return torch.tanh(self.W_x(x) + self.k_h * self.W_h(h) + self.k_c * self.W_c(h))
    def init_hidden(self, batch):
        return torch.zeros(batch, self.hidden_dim, device=self.W_x.weight.device)

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, k_h, k_c):
        super().__init__()
        self.cell = PairedSpectralCell(input_dim, hidden_dim, k_h, k_c)
        self.out = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        B, T, _ = x.shape
        h = self.cell.init_hidden(B)
        outputs = []
        for t in range(T):
            h = self.cell(x[:, t, :], h)
            outputs.append(self.out(h))
        return torch.stack(outputs, dim=1)

# ---------- Training loop ----------
@dataclass
class Result:
    k_h: float
    k_c: float
    final_loss: float
    mean_grad_last50: float
    grad_std_last50: float
    loss_curve: list = field(default_factory=list)
    grad_curve: list = field(default_factory=list)
    diverged: bool = False

def train_one(k_h, k_c, seq_len=5000, n_steps=300, hidden=64, batch=32,
              lr=1e-3, grad_clip=5.0, verbose=False):
    torch.manual_seed(42)
    model = RNNModel(3, hidden, 3, k_h, k_c).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses, gnorms = [], []
    diverged = False
    for step in range(n_steps):
        inp, tgt = generate_lorenz(batch, seq_len, device=device)
        out = model(inp)
        loss = loss_fn(out, tgt)
        if not torch.isfinite(loss):
            diverged = True
            break
        opt.zero_grad()
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip).item()
        opt.step()
        losses.append(loss.item())
        gnorms.append(gnorm)
        if verbose and step % 50 == 0:
            print(f'  step {step}: loss={loss.item():.6f} grad={gnorm:.4f}')
    if diverged or len(losses) < 50:
        return Result(k_h, k_c, float('inf'), float('inf'), float('inf'),
                      losses, gnorms, diverged=True)
    return Result(k_h, k_c,
                  float(np.mean(losses[-20:])),
                  float(np.mean(gnorms[-50:])),
                  float(np.std(gnorms[-50:])),
                  losses, gnorms)

# ---------- Smoke test ----------
print('\n=== Smoke test: (PHI, ALPHA) ===')
t0 = time.time()
r = train_one(PHI, ALPHA, seq_len=2000, n_steps=50, verbose=True)
print(f'Final loss: {r.final_loss:.6f}  |  time: {time.time()-t0:.1f}s')

# ---------- SWEEP A: paired reciprocal (k, 1/k) ----------
SWEEP_A_KS = [0.90, 1.00, 1.10, 1.15, 1.20, 1.2732, 1.30, 1.35, 1.45, 1.60, 1.75]
SEQ_LEN = 5000
N_STEPS = 300

print('\n=== SWEEP A: paired reciprocal (k, 1/k) ===')
results_a = []
for k in SWEEP_A_KS:
    t0 = time.time()
    r = train_one(k_h=k, k_c=1.0/k, seq_len=SEQ_LEN, n_steps=N_STEPS)
    results_a.append(r)
    tag = ' ← 4/π' if abs(k - PHI) < 1e-3 else ('  vanilla' if abs(k-1.0) < 1e-3 else '')
    print(f'k={k:.4f} (1/k={1/k:.4f})  final={r.final_loss:.6f}  '
          f'grad={r.mean_grad_last50:.4f}±{r.grad_std_last50:.4f}  '
          f'{"DIVERGED" if r.diverged else ""} '
          f'[{time.time()-t0:.0f}s]{tag}')

# ---------- SWEEP B: break the reciprocal ----------
SWEEP_B_KCS = [0.40, 0.55, 0.70, 0.7854, 0.85, 1.00, 1.15, 1.2732]
print('\n=== SWEEP B: fixed k_h=PHI, vary k_c ===')
results_b = []
for kc in SWEEP_B_KCS:
    t0 = time.time()
    r = train_one(k_h=PHI, k_c=kc, seq_len=SEQ_LEN, n_steps=N_STEPS)
    results_b.append(r)
    product = PHI * kc
    tag = ' ← reciprocal' if abs(product - 1.0) < 1e-3 else ''
    print(f'k_h=PHI={PHI:.4f}  k_c={kc:.4f}  product={product:.4f}  '
          f'final={r.final_loss:.6f}  grad={r.mean_grad_last50:.4f}  '
          f'{"DIVERGED" if r.diverged else ""} '
          f'[{time.time()-t0:.0f}s]{tag}')

# ---------- Results tables + plots ----------
df_a = pd.DataFrame([{
    'k_h': r.k_h, 'k_c': r.k_c, 'product': r.k_h * r.k_c,
    'final_loss': r.final_loss, 'mean_grad': r.mean_grad_last50,
    'grad_std': r.grad_std_last50, 'diverged': r.diverged,
} for r in results_a])
df_b = pd.DataFrame([{
    'k_h': r.k_h, 'k_c': r.k_c, 'product': r.k_h * r.k_c,
    'final_loss': r.final_loss, 'mean_grad': r.mean_grad_last50,
    'grad_std': r.grad_std_last50, 'diverged': r.diverged,
} for r in results_b])
print('\nSWEEP A — reciprocal pairs (k, 1/k):')
print(df_a.to_string(index=False))
print('\nSWEEP B — fixed k_h=PHI, vary k_c:')
print(df_b.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(df_a['k_h'], df_a['final_loss'], 'o-', color='steelblue')
ax.axvline(PHI, color='crimson', linestyle='--', alpha=0.6, label=f'4/π = {PHI:.4f}')
ax.axvline(1.0, color='gray',   linestyle=':',  alpha=0.6, label='vanilla (k=1)')
ax.set_xlabel('k  (paired with 1/k)'); ax.set_ylabel('Final loss')
ax.set_title('Sweep A: Final loss vs paired-reciprocal k'); ax.set_yscale('log'); ax.legend()

ax = axes[0, 1]
ax.errorbar(df_a['k_h'], df_a['mean_grad'], yerr=df_a['grad_std'], fmt='o-', color='seagreen')
ax.axvline(PHI, color='crimson', linestyle='--', alpha=0.6)
ax.set_xlabel('k'); ax.set_ylabel('Mean grad norm (last 50 steps)')
ax.set_title('Sweep A: Gradient stability vs k')

ax = axes[1, 0]
ax.plot(df_b['k_c'], df_b['final_loss'], 'o-', color='darkorange')
ax.axvline(ALPHA, color='crimson', linestyle='--', alpha=0.6, label=f'π/4 = {ALPHA:.4f} (reciprocal)')
ax.set_xlabel('k_c  (with k_h fixed at 4/π)'); ax.set_ylabel('Final loss')
ax.set_title('Sweep B: Final loss vs k_c (reciprocal broken)')
ax.set_yscale('log'); ax.legend()

ax = axes[1, 1]
ax.plot(df_b['product'], df_b['final_loss'], 'o-', color='purple')
ax.axvline(1.0, color='crimson', linestyle='--', alpha=0.6, label='product = 1.0')
ax.set_xlabel('k_h · k_c  (product)'); ax.set_ylabel('Final loss')
ax.set_title('Sweep B: Loss vs spectral product'); ax.set_yscale('log'); ax.legend()

plt.tight_layout(); plt.savefig('vru_ablation_results.png', dpi=120, bbox_inches='tight'); plt.show()
df_a.to_csv('sweep_a.csv', index=False)
df_b.to_csv('sweep_b.csv', index=False)
print('\nSaved: sweep_a.csv, sweep_b.csv, vru_ablation_results.png')

# ---------- Interpretation helper ----------
losses_a = df_a[~df_a['diverged']]['final_loss']
loss_range = losses_a.max() / losses_a.min() if len(losses_a) > 1 else float('inf')
best_a = df_a.loc[df_a['final_loss'].idxmin()]
best_b = df_b.loc[df_b['final_loss'].idxmin()]
phi_row = df_a[np.isclose(df_a['k_h'], PHI, atol=1e-3)].iloc[0]

print('\nINTERPRETATION')
print('=' * 60)
print(f"Sweep A best: k={best_a['k_h']:.4f} → loss={best_a['final_loss']:.6f}")
print(f"At PHI (4/π): k={phi_row['k_h']:.4f} → loss={phi_row['final_loss']:.6f}")
print(f'Max/min loss ratio across Sweep A (non-diverged): {loss_range:.2f}x')
print(f"Sweep B best k_c: {best_b['k_c']:.4f} (product={best_b['product']:.4f})")
print(f"Diverged runs in Sweep B: {df_b['diverged'].sum()} / {len(df_b)}")
print()
print('Read:')
print('  - Sweep A loss ratio < 2x            →  Outcome B (plateau) — value not special')
print('  - Sweep A ratio > 5x, min at PHI     →  Outcome A (sharp peak)')
print('  - Sweep B: collapse off reciprocal   →  Outcome C supported')
print('  - Sweep B: no sensitivity to product →  Outcome C denied')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
PHI = 4/π  = 1.273240
ALPHA = π/4 = 0.785398
PHI * ALPHA = 1.000000

=== Smoke test: (PHI, ALPHA) ===
  step 0: loss=0.625602 grad=15.5131
Final loss: 0.026327  |  time: 145.5s

=== SWEEP A: paired reciprocal (k, 1/k) ===


KeyboardInterrupt: 

In [3]:
# ============================================================
# VRU Spectral Pairing Ablation — SCALED (fast directional read)
#
# seq_len=1500, n_steps=150. Targets ~20-40 min on T4.
# Same question, same three outcomes — just less compute per run.
# ============================================================

import torch
import torch.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import dataclass, field
import time

# ---------- Setup ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(42); np.random.seed(42)

PHI = 4.0 / math.pi
ALPHA = math.pi / 4.0
print(f'PHI = 4/π  = {PHI:.6f}')
print(f'ALPHA = π/4 = {ALPHA:.6f}')
print(f'PHI * ALPHA = {PHI*ALPHA:.6f}')

# ---------- Lorenz attractor ----------
def generate_lorenz(batch, seq_len, dt=0.01, device='cuda'):
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    state = torch.randn(batch, 3, device=device) * 0.5 + torch.tensor([1.0, 1.0, 1.0], device=device)
    trajectory = torch.zeros(batch, seq_len + 1, 3, device=device)
    trajectory[:, 0] = state
    def f(s):
        x, y, z = s[..., 0], s[..., 1], s[..., 2]
        return torch.stack([sigma*(y-x), x*(rho-z)-y, x*y-beta*z], dim=-1)
    for t in range(seq_len):
        k1 = f(state)
        k2 = f(state + 0.5*dt*k1)
        k3 = f(state + 0.5*dt*k2)
        k4 = f(state + dt*k3)
        state = state + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
        trajectory[:, t+1] = state
    trajectory = trajectory / 30.0
    return trajectory[:, :-1], trajectory[:, 1:]

# ---------- Paired-spectral RNN ----------
class PairedSpectralCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, k_h, k_c):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.k_h, self.k_c = k_h, k_c
        self.W_x = nn.Linear(input_dim, hidden_dim, bias=True)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_c = nn.Linear(hidden_dim, hidden_dim, bias=False)
        nn.init.orthogonal_(self.W_h.weight)
        nn.init.orthogonal_(self.W_c.weight)
    def forward(self, x, h):
        return torch.tanh(self.W_x(x) + self.k_h * self.W_h(h) + self.k_c * self.W_c(h))
    def init_hidden(self, batch):
        return torch.zeros(batch, self.hidden_dim, device=self.W_x.weight.device)

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, k_h, k_c):
        super().__init__()
        self.cell = PairedSpectralCell(input_dim, hidden_dim, k_h, k_c)
        self.out = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        B, T, _ = x.shape
        h = self.cell.init_hidden(B)
        outputs = []
        for t in range(T):
            h = self.cell(x[:, t, :], h)
            outputs.append(self.out(h))
        return torch.stack(outputs, dim=1)

# ---------- Training loop ----------
@dataclass
class Result:
    k_h: float
    k_c: float
    final_loss: float
    mean_grad_last50: float
    grad_std_last50: float
    loss_curve: list = field(default_factory=list)
    grad_curve: list = field(default_factory=list)
    diverged: bool = False

def train_one(k_h, k_c, seq_len=1500, n_steps=150, hidden=48, batch=24,
              lr=1e-3, grad_clip=5.0, verbose=False):
    torch.manual_seed(42)
    model = RNNModel(3, hidden, 3, k_h, k_c).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses, gnorms = [], []
    diverged = False
    for step in range(n_steps):
        inp, tgt = generate_lorenz(batch, seq_len, device=device)
        out = model(inp)
        loss = loss_fn(out, tgt)
        if not torch.isfinite(loss):
            diverged = True
            break
        opt.zero_grad()
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip).item()
        opt.step()
        losses.append(loss.item())
        gnorms.append(gnorm)
        if verbose and step % 30 == 0:
            print(f'  step {step}: loss={loss.item():.6f} grad={gnorm:.4f}')
    if diverged or len(losses) < 30:
        return Result(k_h, k_c, float('inf'), float('inf'), float('inf'),
                      losses, gnorms, diverged=True)
    tail = min(30, len(losses))
    return Result(k_h, k_c,
                  float(np.mean(losses[-tail:])),
                  float(np.mean(gnorms[-tail:])),
                  float(np.std(gnorms[-tail:])),
                  losses, gnorms)

# ---------- Smoke test (small) ----------
print('\n=== Smoke test: (PHI, ALPHA), seq=800, steps=30 ===')
t0 = time.time()
r = train_one(PHI, ALPHA, seq_len=800, n_steps=30, verbose=True)
print(f'Final loss: {r.final_loss:.6f}  |  time: {time.time()-t0:.1f}s')

# ---------- SWEEP A: paired reciprocal (k, 1/k) — trimmed ----------
SWEEP_A_KS = [0.95, 1.00, 1.10, 1.20, 1.2732, 1.35, 1.50, 1.70]
SEQ_LEN = 1500
N_STEPS = 150

print('\n=== SWEEP A: paired (k, 1/k) ===')
results_a = []
for k in SWEEP_A_KS:
    t0 = time.time()
    r = train_one(k_h=k, k_c=1.0/k, seq_len=SEQ_LEN, n_steps=N_STEPS)
    results_a.append(r)
    tag = ' ← 4/π' if abs(k - PHI) < 1e-3 else ('  vanilla' if abs(k-1.0) < 1e-3 else '')
    print(f'k={k:.4f} (1/k={1/k:.4f})  final={r.final_loss:.6f}  '
          f'grad={r.mean_grad_last50:.4f}±{r.grad_std_last50:.4f}  '
          f'{"DIVERGED" if r.diverged else ""} '
          f'[{time.time()-t0:.0f}s]{tag}')

# ---------- SWEEP B: break the reciprocal — trimmed ----------
SWEEP_B_KCS = [0.40, 0.60, 0.7854, 1.00, 1.2732]
print('\n=== SWEEP B: k_h=PHI, vary k_c ===')
results_b = []
for kc in SWEEP_B_KCS:
    t0 = time.time()
    r = train_one(k_h=PHI, k_c=kc, seq_len=SEQ_LEN, n_steps=N_STEPS)
    results_b.append(r)
    product = PHI * kc
    tag = ' ← reciprocal' if abs(product - 1.0) < 1e-3 else ''
    print(f'k_h=PHI={PHI:.4f}  k_c={kc:.4f}  product={product:.4f}  '
          f'final={r.final_loss:.6f}  grad={r.mean_grad_last50:.4f}  '
          f'{"DIVERGED" if r.diverged else ""} '
          f'[{time.time()-t0:.0f}s]{tag}')

# ---------- Tables + plots ----------
df_a = pd.DataFrame([{
    'k_h': r.k_h, 'k_c': r.k_c, 'product': r.k_h * r.k_c,
    'final_loss': r.final_loss, 'mean_grad': r.mean_grad_last50,
    'grad_std': r.grad_std_last50, 'diverged': r.diverged,
} for r in results_a])
df_b = pd.DataFrame([{
    'k_h': r.k_h, 'k_c': r.k_c, 'product': r.k_h * r.k_c,
    'final_loss': r.final_loss, 'mean_grad': r.mean_grad_last50,
    'grad_std': r.grad_std_last50, 'diverged': r.diverged,
} for r in results_b])
print('\nSWEEP A:'); print(df_a.to_string(index=False))
print('\nSWEEP B:'); print(df_b.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

ax = axes[0, 0]
ax.plot(df_a['k_h'], df_a['final_loss'], 'o-', color='steelblue')
ax.axvline(PHI, color='crimson', linestyle='--', alpha=0.6, label=f'4/π = {PHI:.4f}')
ax.axvline(1.0, color='gray', linestyle=':', alpha=0.6, label='vanilla')
ax.set_xlabel('k (paired with 1/k)'); ax.set_ylabel('Final loss')
ax.set_title('Sweep A: Loss vs paired-reciprocal k'); ax.set_yscale('log'); ax.legend()

ax = axes[0, 1]
ax.errorbar(df_a['k_h'], df_a['mean_grad'], yerr=df_a['grad_std'], fmt='o-', color='seagreen')
ax.axvline(PHI, color='crimson', linestyle='--', alpha=0.6)
ax.set_xlabel('k'); ax.set_ylabel('Mean grad norm')
ax.set_title('Sweep A: Grad stability vs k')

ax = axes[1, 0]
ax.plot(df_b['k_c'], df_b['final_loss'], 'o-', color='darkorange')
ax.axvline(ALPHA, color='crimson', linestyle='--', alpha=0.6, label=f'π/4 (reciprocal)')
ax.set_xlabel('k_c (k_h fixed at 4/π)'); ax.set_ylabel('Final loss')
ax.set_title('Sweep B: Loss vs k_c'); ax.set_yscale('log'); ax.legend()

ax = axes[1, 1]
ax.plot(df_b['product'], df_b['final_loss'], 'o-', color='purple')
ax.axvline(1.0, color='crimson', linestyle='--', alpha=0.6, label='product = 1.0')
ax.set_xlabel('k_h · k_c'); ax.set_ylabel('Final loss')
ax.set_title('Sweep B: Loss vs spectral product'); ax.set_yscale('log'); ax.legend()

plt.tight_layout(); plt.savefig('vru_ablation_scaled.png', dpi=120, bbox_inches='tight'); plt.show()
df_a.to_csv('sweep_a_scaled.csv', index=False)
df_b.to_csv('sweep_b_scaled.csv', index=False)

# ---------- Interpretation ----------
losses_a = df_a[~df_a['diverged']]['final_loss']
loss_range = losses_a.max() / losses_a.min() if len(losses_a) > 1 else float('inf')
best_a = df_a.loc[df_a['final_loss'].idxmin()]
best_b = df_b.loc[df_b['final_loss'].idxmin()]
phi_row = df_a[np.isclose(df_a['k_h'], PHI, atol=1e-3)].iloc[0]

print('\nINTERPRETATION (scaled read)')
print('=' * 60)
print(f"Sweep A best: k={best_a['k_h']:.4f} → loss={best_a['final_loss']:.6f}")
print(f"At PHI (4/π): k={phi_row['k_h']:.4f} → loss={phi_row['final_loss']:.6f}")
print(f'Max/min loss ratio (non-diverged): {loss_range:.2f}x')
print(f"Sweep B best k_c: {best_b['k_c']:.4f} (product={best_b['product']:.4f})")
print(f"Diverged in Sweep B: {df_b['diverged'].sum()}/{len(df_b)}")
print()
print('Directional read:')
print('  - A ratio < 2x            →  Outcome B (plateau)')
print('  - A ratio > 5x, min at PHI →  Outcome A (sharp peak)')
print('  - B collapses off recip   →  Outcome C supported')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
PHI = 4/π  = 1.273240
ALPHA = π/4 = 0.785398
PHI * ALPHA = 1.000000

=== Smoke test: (PHI, ALPHA), seq=800, steps=30 ===


KeyboardInterrupt: 